In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/guki/.conda/envs/miniminds/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 思考题
1. 为什么小模型要用小词表？（提示：Embedding 层占总参数的比例）
回答：词表越大，embedding层占用的参数量越大，训练/推理的时候占用的显存越大。6400 应该是实践中找到的平衡点.
Embedding 占用的内容不过多，Transformer的结构占用的会多。
2. BPE 和 WordPiece 的区别是什么？
回答：BPE是找到最频繁的字符对，仅关注频率。WordPice找的是最大似然对数，可能保存更多的语义？实际的应用中差距不大

In [3]:
tok = AutoTokenizer.from_pretrained("../model_learn_tokenizer")
print(tok.encode("你好世界"))
print(tok.decode([336, 550, 913]))
print(len(tok.vocab))
message = [{"role": "user", "content": "你使用的是什么模型?"}, {"role": "system", "content": "我使用的是一个基于transformer架构的语言模型，专门设计用于生成自然语言文本。"}]
print(tok.apply_chat_template(message, return_tensors="pt", padding=True, truncation=True))
print(tok.decode([   1,  120,  118, 6237,  234,  336, 1014,  296, 1336,  751,   66,    2,
          234,    1,  118,  124,  118, 4348,  112,  234,  328, 1014, 1442,  697,
         1015,  119, 2440,  113,  118,  105, 4769, 3119,  117, 4002,  296,  957,
          751,  294,  764,  965, 1058, 2486,  937, 2888, 2443,  301,    2,  234]))

[336, 550, 913]
你好世界
6400


第二阶段

tensor([[   1,  120,  118, 6237,  234,  336, 1014,  296, 1336,  751,   66,    2,
          234,    1,  118,  124,  118, 4348,  112,  234,  328, 1014, 1442,  697,
         1015,  119, 2440,  113,  118,  105, 4769, 3119,  117, 4002,  296,  957,
          751,  294,  764,  965, 1058, 2486,  937, 2888, 2443,  301,    2,  234]])
<|im_start|>user
你使用的是什么模型?<|im_end|>
<|im_start|>system
我使用的是一个基于transformer架构的语言模型，专门设计用于生成自然语言文本。<|im_end|>



In [ ]:
## 点积注意力机制的联系 最好能找到数学的原理或者直觉
import torch
import torch.nn as nn 
x1 = torch.randn(2, 3, 4) # batch_size, seq_len, hidden_size
x2 = torch.randn(2, 5, 4)

# 计算原始权重
raw_weights = torch.bmm(x1, x2.transpose(1, 2))# batch_size, seq_len1, seq_len2
print(raw_weights.shape)

# 使用softmax归一化权重
weights = torch.softmax(raw_weights, dim=2) # batch_size, seq_len1, seq_len2
print(weights.shape)

# 将注意力权重与 X2相乘  得到 x1对X2的注意力表示
attention_output = torch.bmm(weights, x2) # batch_size, seq_len1,
print(attention_output)

# 点积是什么 |A|*|B|*cos(theta) 其中theta是A和B之间的夹角，点积越大，说明A和B越相相似
# 归一化之后相当于 |A|*|B| / |A|*|B| 的这种感觉，也就是主要关注cos(theta)（模长也起作用） 也就是说 多维矩阵中的每一个值都是 A_i 与 B_i 向量之间的相似归一化相似度量值 而且 A_1B_1 + A_1B_2 + xxx+A_1B_n = 1,  还有和各个向量之间的相似度相对值 
# 归一化之后的结果* X2 做的也是点乘，相当于 对 X1 在对于X2的每个位置的相似度的加权求和，也就是一种，基于对X2的相似度的，X1的重计算的感觉。也就是注意力？

# AI的回答
# 你的大方向理解是对的，但有几个关键误区需要纠正。
# 先说对的部分
# 核心直觉完全正确： 注意力 = 根据相似度做加权求和
# attention_output[i] = Σ_j (weight[i,j] * X2[j])
# 每个 attention_output[i] 都是 X2 所有位置的加权混合，权重由 X1[i] 与 X2[j] 的相似度决定。这就是"注意力"的本质。
# ---
# 需要纠正的地方
# 误区 1：归一化 ≠ 只看夹角
# 你说"归一化之后主要关注 cos(theta)"，这是错的。
# softmax 不是 L2 归一化，不是 |A|*|B| / (|A|*|B|)
# softmax 是指数函数:  exp(score) / Σ exp(score_j)
#                   =  exp(|Q|·|K|·cosθ) / Σ exp(|Q|·|K_j|·cosθ_j)
# 向量的模长 |Q| 和 |K| 仍然在指数里起作用。 这正是为什么 Transformer 要做 scale：
# scores = Q @ K.T / sqrt(d_k)    # 除以 sqrt(d_k) 就是防止大模长让 softmax 太尖锐
# 如果不除：|Q| 大 → 所有 score 都很大 → softmax 趋近 one-hot → 梯度消失。
# 误区 2：A1·B1 + A1·B2 + ... = 1
# 不是原始点积和为 1，是 softmax 之后和为 1：
# softmax([Q₁·K₁, Q₁·K₂, Q₁·K₃]) = [0.6, 0.3, 0.1]
#   ↑ 这些加起来 = 1
#   ↑ 原始点积加起来 ≠ 1
# 误区 3："X1 的重计算"
# 反了。不是 X1 被重算，是 X2 被 X1 的视角重新组装：
# attention_output 的形状和 X1 一样 (2, 3, 4)
# 但内容 = X2 的 3 种加权混合
# 更准确的说法：
# X1 是"问题"，X2 是"知识库"
# attention_output = X1 从 X2 里检索到的信息
# ---
# 直观数学直觉
#           Q (query)              K (key)
#       ┌─────────────┐       ┌─────────────┐
#       │  "苹果"      │       │  "苹果": 好吃  │
#       │  "香蕉"      │  ×    │  "香蕉": 贵   │   → softmax → 加权 → V 混合
#       │  "橘子"      │       │  "橘子": 便宜  │
#       └─────────────┘       └─────────────┘
#             Q·K^T                  V (value)
#       相似度矩阵              实际要检索的信息
# 点积大 → 相似度大 → softmax 权重大 → 对应 V 被更多地取出来。
# ---
# 对应到 MiniMind 的代码
# 你写的裸注意力对应 model_minimind.py L110-133：
# # L112: Q, K, V 分别由线性层投影
# xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
# # L124-125: scaled dot product attention (Flash)
# F.scaled_dot_product_attention(xq, xk, xv)
# # L127-130: 手动 fallback（等价于你写的代码，但加了 scale 和 causal mask）
# scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)  # ← 注意除以 sqrt(d_k)
# scores += causal_mask  # ← 自回归只允许看左边
# output = softmax(scores) @ xv
# 和你写的代码就差两件事：
# 1. 除以 sqrt(d_k) — 防止 softmax 饱和
# 2. Causal mask — 防止偷看未来


# 让我理解下attention 机制 和 大模型上的注意力机制
# 1. 注意力机制 Q K V ， Q 和 K 用于计算注意力机制，V用于提供值，Q K 进行矩阵运算之后，得到每个位置对于其他位置的注意力分数，其实也是一种相似度度量，然后以这个分数和V进行点乘，得到每个位置对于V的注意力表示，V是源输入的一个投影
# 在大模型上 由于语言模型是知道前面的值，预测下一个值 所以需要添加一个casual mask 避免 注意力机制能获取到未来的注意力信息
# 关于其他的运算其实是做维度的匹配 
# 然后RoPE编码，是一种位置编码，用于表示位置之间的关系，用虚数运算模拟旋转，来表示位置之间的关系。
# K V cache 是推理的时候使用的但是为什么需要知道 K V 呢而不是Q这一点需要看一下 因为K V 是在训练的时候计算的，而Q 是在推理的时候计算的，所以需要知道K V 来计算注意力分数 K 和 V 其实是历史+ 现在 Q *K 得到当前对于当前和历史的注意力分数，和V运算得到最终的注意力结果表示
# V Cache 确实是一个不断增长的缓存：
# Step 1: 算 K₁,V₁     → cache = [K₁, V₁]
# Step 2: 算 K₂,V₂     → cache = [K₁, V₁, K₂, V₂]
# Step 3: 算 K₃,V₃     → cache = [K₁, V₁, K₂, V₂, K₃, V₃]
# 到第 1000 步时，cache 里累计存了 1000 个 K 和 1000 个 V，不是存了 1000 份独立的 cache。
# 重新对比
 
# Step 1000 的计算
# Step 1000 的 cache
# 之前说的公式 2 × seq_len × n_kv_heads × head_dim × dtype 就是到第 n 步时这个 cache 累计占用的总空间，不是每步额外乘 n。
# 之前表述不清，感谢纠正。

torch.Size([2, 3, 5])
torch.Size([2, 3, 5])
tensor([[[ 0.5157,  0.7630, -0.9430,  1.1085],
         [ 0.2931,  0.0984,  0.1260, -1.5729],
         [ 0.7096,  0.4114, -0.3955,  0.3026]],

        [[-1.3010,  1.7322, -0.5213,  1.1990],
         [-0.5465,  0.0851,  0.5760,  1.0244],
         [-0.9382,  1.4342, -0.3334,  0.7220]]])


# 关于注意力机制的理解
当前理解评估
✅ 已经掌握的部分
概念
Q×K 得注意力分数
Causal mask
KV Cache 本质
RoPE
推理时 KV 是动态增长的
⚠️ 还需要补强的部分
概念
Scaled 的意义
Multi-Head Attention
训练 vs 推理的并行差异
快速自测
如果下面 3 个问题你能不查代码答出来，说明这部分可以过了：
1. 为什么注意力分数要除以 sqrt(d_k)，不除会怎样？
2. MiniMind 用 GQA（Q=8头, KV=4头），为什么 KV 头数可以少于 Q？重复 K/V 怎么实现？
3. 训练时 batch_size=32, seq_len=1000，一个 forward pass 会算多少次 QKV？推理时生成 1000 个 token 呢？
▣  Build · Qwen3.6 Plus · 12.7s
sqrt(dk)是为了减少维度以及极端值对注意力分数计算的影响，极端值导致忽略了其他的注意力。
Thinking: The user answered the first self-test question. Let me evaluate their answer and continue the discussion.
User's answer: "sqrt(dk)是为了减少维度以及极端值对注意力分数计算的影响，极端值导致忽略了其他的注意力。"
Translation: "sqrt(dk) is to reduce the impact of dimensionality and extreme values on the attention score calculation. Extreme values cause the model to ignore other attention."
This is essentially correct. Let me evaluate:
1. Correct part: Yes, dividing by sqrt(d_k) prevents extreme values. When Q and K are initialized with mean=0, variance=1, the dot product has mean=0 and variance=d_k. So as d_k increases, the variance of Q·K increases, leading to larger absolute values. After softmax, large values become ~1, small values become ~0 → gradients vanish.
2. Missing precision: The user said "减少维度" which is slightly misleading - it doesn't reduce dimension, it normalizes the variance. The dot product's variance is d_k, so dividing by sqrt(d_k) brings variance back to 1.
3. "极端值导致忽略了其他的注意力" - This is the right intuition. Large scores → softmax becomes one-hot → gradients vanish for non-max positions.
Let me provide a concise evaluation and move to the next questions.
基本正确，但"减少维度"这个说法不太准确。
精确理解
Q ~ N(0, 1), K ~ N(0, 1)
Q·K 的方差 = d_k          ← 维度越大，方差越大
不除: d_k=768 时，Q·K 范围 [-100, 100]
     → softmax 趋近 [0, 0, ..., 1, ..., 0]
     → 梯度消失
除以 sqrt(d_k): Q·K/sqrt(d_k) 的方差 ≈ 1
     → softmax 分布均匀，梯度健康
不是"减少维度"，是把方差归一化回 1，跟维度无关。
---
继续回答剩下的两个：
2. MiniMind Q=8头 KV=4头，为什么可以少于 Q？重复 K/V 怎么实现？
3. 训练时 batch=32, seq_len=1000，一次 forward 算多少次 QKV？推理生成 1000 个 token 呢？
▣  Build · Qwen3.6 Plus · 20.0s
KV 可以复制，。通过矩阵的复制 来实现 K V 的升维
Thinking: The user is answering the second self-test question.
They said: "KV can be copied. By matrix copying to achieve K V's dimension upgrade."
This refers to Grouped Query Attention (GQA) where repeat_kv is used.
I should confirm this is correct, briefly show the code snippet from model_minimind.py that implements this (repeat_kv), and then ask the third question again if they want to continue testing, or just validate it.
The user's answer is essentially correct.
- Q has 8 heads.
- KV has 4 heads.
- n_rep = 8 // 4 = 2.
- Each K/V head is repeated 2 times to match Q's dimension for the dot product.
I will validate this and mention the function name repeat_kv in the code.
I will also briefly prompt for the 3rd question if they haven't answered it yet.
Wait, looking at the previous turn, I asked 3 questions.
1. sqrt(dk) - User answered.
2. GQA/Repeat KV - User answered "KV can be copied...".
3. Training vs Inference calculation counts - Not answered yet.
So I should confirm question 2 and ask for question 3.
正确。对应代码 model_minimind.py 里的 repeat_kv 函数 (L85-88)：
def repeat_kv(x, n_rep):
    # x shape: (batch, seq_len, n_kv_heads, head_dim)
    # 把每个 head 复制 n_rep 次，扩成和 Q 一样的头数
    return x[:, :, :, None, :].expand(..., n_rep, ...).reshape(...)
比如 MiniMind 的 Q=8, KV=4 → n_rep=2 → 每个 KV head 复制一次变 2 份。
---
还有第 3 题：
训练时 batch=32, seq_len=1000，一次 forward pass 算多少次 QKV 的计算（即多少个 token 的投影）？
推理时生成 1000 个 token 呢？
▣  Build · Qwen3.6 Plus · 11.6s
32次 Q K V 的计算，生成1000个token Q 计算1000次，K V 是在之前的基础上“累加” 1000次
Thinking: The user is answering question 3 about the computational cost of training vs inference in transformers. Let me analyze their answer.
Training: batch=32, seq_len=1000
User says: "32次 Q K V 的计算" - This is wrong. In training, for each sample in the batch, all tokens' QKV are computed. So it's 32 × 1000 = 32,000 tokens worth of QKV projection.
Inference: generate 1000 tokens
User says: "Q 计算1000次，K V 是在之前的基础上'累加' 1000次" - This is essentially correct but a bit ambiguous. Q is computed 1000 times (once per generation step). K and V are also computed 1000 times (once per step for the new token), and they are accumulated in the cache.
Let me correct the training part.
第 2 题对了。第 3 题的推理部分对，但训练部分错了。
训练时
不是一次只算 32 次 QKV，而是 32 × 1000 = 32,000 个 token 的 QKV 一次性并行算完。
输入: 32 条句子，每条 1000 个 token
    ↓ 一次矩阵乘法
Q = [batch=32, seq_len=1000, hidden=768] → (32×1000, 768) @ (768, 768) → [32, 1000, 768]
K = 同上
V = 同上
注意力: Q @ Kᵀ  → [32, 1000, 1000]   ← 这一步就是 O(n²)
训练时所有 Q、K、V 是同时存在的，一次 matmul 搞定。
推理时
Step 1: 算 1 个 token 的 Q, K, V
Step 2: 算 1 个 token 的 Q, K, V → K,V 追加到 cache
Step 3: 算 1 个 token 的 Q, K, V → K,V 追加到 cache
...
Step 1000: 算 1 个 token 的 Q, K, V → K,V 追加到 cache
总计: Q 算了 1000 次，K 算了 1000 次，V 算了 1000 次
      注意力每次都和累积的 cache 算 → [1, seq_len_i] 的 score
---
核心对比
训练: 32 × 1000 个 QKV 一次算完 → 1 次 attention
推理: 每次算 1 个 QKV → 1000 次 attention，每次用累积的 cache



我有一个问题，K V 可以升维，那Q也可以升维啊，从数学运算的角度Q的维度也可以少于K V
Thinking: 用户提出了一个很好的问题：既然 K,V 可以通过 repeat_kv 复制来匹配 Q 的维度，那反过来 Q 少于 K,V 是否也可以？这涉及 GQA 和 MQA 的设计原理。
数学上确实可以，但不需要，因为 Q 少 K,V 多等价于 GQA 的逆向，没有计算收益。
为什么只减少 KV 头，不减少 Q 头
核心原因：注意力的计算量取决于 Q 头数
Attention 计算量:
  Q @ Kᵀ → [n_q_heads, seq_len, seq_len]
  weight @ V → [n_q_heads, seq_len, head_dim]
                    ↑ 输出通道数 = Q 头数
- Q 头数 决定输出维度 → 影响后续所有层的计算量
- KV 头数 只影响 Q@Kᵀ 时 K 的复制成本（极低）
Q=8, KV=4 (MiniMind GQA):  输出 8 个 head → 正常
Q=4, KV=8 (反向):          输出 4 个 head → 表达能力变弱，还多了 KV 的计算浪费
极端对比
 
Q 头数
KV 头数
输出 head
KV cache
表达能力
KV 少 → 省显存（cache 小）+ 省 Q@Kᵀ 的计算，同时输出 head 不变。
Q 少 → 输出维度少，表达能力直接下降，得不偿失。